# Case study: Combining multiple sources

**A common task in data manipulation is combining or merging multiple datasets.**

This case study will work with data taken from [NHS England's open diagnostic imaging dataset from 2019/20.](https://www.england.nhs.uk/statistics/statistical-work-areas/diagnostic-imaging-dataset/) 

**We will use three datasets**

* Diagnostic imaging referrals by provider
* Diagnostic imaging waiting times by provider
* Diagnostic imaging report times by provider.

We are only interested in the annual figures and will combine these into a single dataset. The formatting in the datasets also has a few minor (and annoying) issues, including how missing data is stored, that we will need to sort out before we can combine. The good news is that `pandas` makes this relatively painless.

After the preprocessing we will create a subset of the data for the South West of England and save it to file.

We will use method chaining to create and preprocess the three datasets into a clean `DataFrame`

> In this instance the datasets are ordered the same. So the same provider and diagnostic imaging type appears in the same row across the datasets. So the task could be completed by preprocessing and then **concatenation** by row. **BUT** this might NOT be correct; resulting in a silent failure and misaligned data! A safer way to do this is to use `pandas.DataFrame.merge()`, joining explicitly on `region`, `org_code` and `imaging_type` rather than relying on row order.  For completeness we will demonstrate both.

## Imports

In [1]:
import pandas as pd
import numpy as np

## Helper functions

As `pandas` method chaining will be used and also that we need to limit the data extracted from each dataframe three helper functions will be created. These accept a `DataFrame` as a parameter and returns a subset of columns.

> In method chaining we will use these with the method `.pipe()`. Don't worry all should become clear when you see the code.

In [2]:
def extract_index_plus_annual_column(df):
    """Extract 5 columns from the original dataset"""
    return df[['region', 'org_code', 'provider', 'imaging_type', 'n_referrals']]

In [3]:
def extract_annual_column_rtt(df):
    """Extract the org code, imaging type and median rtt over a year"""
    return df[['region', 'org_code', 'imaging_type', 'provider', 'mdn_days_rtt']]

In [4]:
def extract_annual_column_ttr(df):
    """Extract the org code, imaging type and median ttr over a year"""
    return df[['region', 'org_code', 'imaging_type', 'provider', 'mdn_days_ttr']]

## Data URLS

In [5]:
# dataset 1: number of referrals
NREFS_URL = 'https://raw.githubusercontent.com/health-data-science-OR' \
    + '/hpdm139-datasets/main/di_counts.csv'

# dataset 2: diagnostic imaging waiting times by provider
TEST_WAIT_URL = 'https://raw.githubusercontent.com/health-data-science-OR' \
    + '/hpdm139-datasets/main/di_rq_to_test.csv'

# dataset 3: diagnostic imaging reporting times by provider
REPT_WAIT_URL = 'https://raw.githubusercontent.com/health-data-science-OR' \
    + '/hpdm139-datasets/main/di_test_to_report.csv'

## A look at the individual datasets

In [6]:
nrefs = pd.read_csv(NREFS_URL)
nrefs.head()

,Region,Org Code,Provider name,Modality,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,Jan,Feb,Mar,Year
0,Y56,NT9,Alliance Medical,Computerized Axial Tomography,495,440,495,445,400,400,475,425,390,450,420,310,"5,145"
1,Y56,NT9,Alliance Medical,Diagnostic Ultrasonography,155,170,120,225,265,220,215,255,230,260,240,135,"2,495"
2,Y56,NT9,Alliance Medical,Magnetic Resonance Imaging,"1,480","1,480","1,365","1,520","1,530","1,485","1,645","1,575","1,515","1,870","1,755","1,515","18,740"
3,Y56,NT9,Alliance Medical,Nuclear Medicine Procedure,.,.,.,.,.,.,.,.,.,*,.,.,*
4,Y56,NT9,Alliance Medical,Plain Radiography,65,105,60,75,85,135,105,65,75,100,85,45,990


In [7]:
rtt = pd.read_csv(TEST_WAIT_URL)
rtt.head()

,Region,Org Code,Provider name,Modality,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,Jan,Feb,Mar,Year
0,Y56,NT9,Alliance Medical,Computerized Axial Tomography,3,2,2,1,2,2,1,1,2,1,1,1,1
1,Y56,NT9,Alliance Medical,Diagnostic Ultrasonography,7,6,6,5,7,10,11,9,17,20,14,8,10
2,Y56,NT9,Alliance Medical,Magnetic Resonance Imaging,7,6,7,6,6,5,6,7,7,7,7,9,6
3,Y56,NT9,Alliance Medical,Nuclear Medicine Procedure,.,.,.,.,.,.,.,.,.,*,.,.,*
4,Y56,NT9,Alliance Medical,Plain Radiography,50,51,56,51,60,57,44,41,45,51,37,37,50


In [8]:
ttr = pd.read_csv(REPT_WAIT_URL)
ttr.head()

,Region,Org Code,Provider name,Modality,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,Jan,Feb,Mar,Year
0,Y56,NT9,Alliance Medical,Computerized Axial Tomography,6,4,5,5,5,4,4,5,5,5,5,4,5
1,Y56,NT9,Alliance Medical,Diagnostic Ultrasonography,0,0,0,0,0,0,0,0,0,0,0,1,0
2,Y56,NT9,Alliance Medical,Magnetic Resonance Imaging,4,4,4,3,5,4,2,3,4,4,3,2,3
3,Y56,NT9,Alliance Medical,Nuclear Medicine Procedure,.,.,.,.,.,.,.,.,.,*,.,.,*
4,Y56,NT9,Alliance Medical,Plain Radiography,.,.,.,.,.,.,.,.,.,.,.,.,.


## Download and pre-processing code.

### Number of referrals.

In [9]:
# organisation info and num referrals
nrefs = (
    pd.read_csv(NREFS_URL)
    .rename(columns={' Year ': 'n_referrals',
                      'Region': 'region',
                      'Org Code': 'org_code',
                      'Provider name': 'provider',
                      'Modality': 'imaging_type'})
    # strip out white space
    .assign(org_code=lambda x: x['org_code'].str.strip(),
            imaging_type=lambda x: x['imaging_type'].str.strip(),
            provider=lambda x: x['provider'].str.strip(),
            region=lambda x: x['region'].str.strip())
    .pipe(extract_index_plus_annual_column)
    .replace(['*', ' * '], np.nan)
    .assign(n_referrals=lambda x: x['n_referrals'].str.strip())
    .assign(n_referrals=lambda x: x['n_referrals'].str.replace(',', ''))
    # deal with NaN - to float and then nullable int (or keep float)
    # NaN is a float. Nullable int use pd.Int64Dtype()
    .astype({'n_referrals': 'float'})
    .astype({'n_referrals': pd.Int32Dtype()})
)

In [10]:
nrefs.head(2)

,region,org_code,provider,imaging_type,n_referrals
0,Y56,NT9,Alliance Medical,Computerized Axial Tomography,5145
1,Y56,NT9,Alliance Medical,Diagnostic Ultrasonography,2495


### Request to diagnostic imaging.

In [11]:
# days from request to imaging
rtt = (
    pd.read_csv(TEST_WAIT_URL)
    .rename(columns={' Year ': 'mdn_days_rtt',
                     'Region': 'region',
                      'Org Code': 'org_code',
                      'Provider name': 'provider',
                      'Modality': 'imaging_type'})
    .assign(org_code=lambda x: x['org_code'].str.strip(),
            imaging_type=lambda x: x['imaging_type'].str.strip(),
            provider=lambda x: x['provider'].str.strip(),
            region=lambda x: x['region'].str.strip())
    .pipe(extract_annual_column_rtt)
    .replace(['*', ' * ', '.'], np.nan)
    .astype({'mdn_days_rtt': 'float'})
)

In [12]:
rtt.head(2)

,region,org_code,imaging_type,provider,mdn_days_rtt
0,Y56,NT9,Computerized Axial Tomography,Alliance Medical,1.0
1,Y56,NT9,Diagnostic Ultrasonography,Alliance Medical,10.0


### Imaging to reporting of results

In [13]:
# days from imaging to report
ttr = (
    pd.read_csv(REPT_WAIT_URL)
    .rename(columns={' Year ': 'mdn_days_ttr',
                  'Region': 'region',
                  'Org Code': 'org_code',
                  'Provider name': 'provider',
                  'Modality': 'imaging_type'})
    .assign(org_code=lambda x: x['org_code'].str.strip(),
            imaging_type=lambda x: x['imaging_type'].str.strip(),
            provider=lambda x: x['provider'].str.strip(),
            region=lambda x: x['region'].str.strip())
    .pipe(extract_annual_column_ttr)
    .replace(['*', ' * ', '.'], np.nan)
    .astype({'mdn_days_ttr': 'float'})
)

In [14]:
ttr.head(2)

,region,org_code,imaging_type,provider,mdn_days_ttr
0,Y56,NT9,Computerized Axial Tomography,Alliance Medical,5.0
1,Y56,NT9,Diagnostic Ultrasonography,Alliance Medical,0.0


## Combine the three sources (risky method: `concat`)

In [15]:
# combined dataset
imaging_df = pd.concat([nrefs, rtt, ttr], axis=1)
imaging_df.head()

,region,org_code,provider,imaging_type,n_referrals,region,org_code,imaging_type,provider,mdn_days_rtt,region,org_code,imaging_type,provider,mdn_days_ttr
0,Y56,NT9,Alliance Medical,Computerized Axial Tomography,5145,Y56,NT9,Computerized Axial Tomography,Alliance Medical,1.0,Y56,NT9,Computerized Axial Tomography,Alliance Medical,5.0
1,Y56,NT9,Alliance Medical,Diagnostic Ultrasonography,2495,Y56,NT9,Diagnostic Ultrasonography,Alliance Medical,10.0,Y56,NT9,Diagnostic Ultrasonography,Alliance Medical,0.0
2,Y56,NT9,Alliance Medical,Magnetic Resonance Imaging,18740,Y56,NT9,Magnetic Resonance Imaging,Alliance Medical,6.0,Y56,NT9,Magnetic Resonance Imaging,Alliance Medical,3.0
3,Y56,NT9,Alliance Medical,Nuclear Medicine Procedure,<NA>,Y56,NT9,Nuclear Medicine Procedure,Alliance Medical,NaN,Y56,NT9,Nuclear Medicine Procedure,Alliance Medical,NaN
4,Y56,NT9,Alliance Medical,Plain Radiography,990,Y56,NT9,Plain Radiography,Alliance Medical,50.0,Y56,NT9,Plain Radiography,Alliance Medical,NaN


In [16]:
imaging_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1325 entries, 0 to 1324
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   region        1325 non-null   object 
 1   org_code      1325 non-null   object 
 2   provider      1325 non-null   object 
 3   imaging_type  1325 non-null   object 
 4   n_referrals   1303 non-null   Int32  
 5   region        1325 non-null   object 
 6   org_code      1325 non-null   object 
 7   imaging_type  1325 non-null   object 
 8   provider      1325 non-null   object 
 9   mdn_days_rtt  1273 non-null   float64
 10  region        1325 non-null   object 
 11  org_code      1325 non-null   object 
 12  imaging_type  1325 non-null   object 
 13  provider      1325 non-null   object 
 14  mdn_days_ttr  1281 non-null   float64
dtypes: Int32(1), float64(2), object(12)
memory usage: 151.5+ KB


In [17]:
imaging_df.shape

(1325, 15)

## Introduction to `Dataframe.merge()`

Before we merge the main three datasets let's create two tiny toy DataFrames and test different types of joins.

A key concept to recognise when merging dataframes is one of the dataframes is to the **left** and the other dataframe is too the **right**. 

The pandas `merge` methods looks like this:

```python
left_df.merge(right, how='inner', on=None, left_on=None, right_on=None, left_index=False, right_index=False, sort=False, suffixes=('_x', '_y'), copy=<no_default>, indicator=False, validate=None)
```

Here the key parameters we will experiment with are `left_df`, `right`, `how`, and `on`.  We will create a **providers** datasets as for the `left_df` and a **waits** dataset for the right.  You can find more details on `merge` in the official [documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.merge.html)

Each row in waits matches exactly one provider, and we join on `org_code` so pandas can find the match no matter what order the rows are in.

In [18]:
providers = pd.DataFrame({
    'org_code': ['REF', 'RK9', 'RA4'],
    'provider': ['Royal Cornwall', 'Gloucestershire', 'Somerset']
})

waits = pd.DataFrame({
    'org_code': ['RK9', 'REF', 'RA4'],   # different order - no problem!
    'mdn_days_rtt': [6, 3, 2]
})

providers.merge(waits, on='org_code')

,org_code,provider,mdn_days_rtt
0,REF,Royal Cornwall,3
1,RK9,Gloucestershire,6
2,RA4,Somerset,2


The join worked even though waits is in a different order.  That's the key advantage over concatenation. 

Now let's try an example that is more complicated. Real data rarely lines up neatly like in our example.  The version below has a provider with no waiting time and a duplicated `org_code`. Watch what happens to both.

In [19]:
providers = pd.DataFrame({
    'org_code': ['REF', 'RK9', 'RA4', 'R0A'],
    'provider': ['Royal Cornwall', 'Gloucestershire', 'Somerset', 'Taunton']
})

waits = pd.DataFrame({
    'org_code': ['REF', 'RK9', 'RA4', 'REF'],   # REF appears twice!
    'mdn_days_rtt': [3, 6, 2, 9]
})


# R0A has disappeared in this frame!
providers.merge(waits, on='org_code')

,org_code,provider,mdn_days_rtt
0,REF,Royal Cornwall,3
1,REF,Royal Cornwall,9
2,RK9,Gloucestershire,6
3,RA4,Somerset,2


Why did R0A disappear? `merge()` defaults to an **inner join**: only rows whose key appears in both tables are kept. R0A exists in providers but not in waits, so it is silently dropped with no warning, no error. This matters: if we had used the default on the real data, every provider–imaging type combination with missing waiting times would have vanished from our analysis.

To keep every provider we use a left join (`how='left'`): all rows from the left table survive, and columns from the right table are filled with NaN where there is no match.

In [20]:
left = providers.merge(waits, on='org_code', how='left')
left

,org_code,provider,mdn_days_rtt
0,REF,Royal Cornwall,3.0
1,REF,Royal Cornwall,9.0
2,RK9,Gloucestershire,6.0
3,RA4,Somerset,2.0
4,R0A,Taunton,NaN


## Combine the three sources (low risk recommended method: `merge`)

Now that we understand `merge` on a small example let's apply it to our three datasets. 

The cells below rebuild the same combined dataset using `pandas.DataFrame.merge()`, joining explicitly on `region`, `org_code` and `imaging_type` rather than relying on row order. This is safer for data manipulation because:

* it works no matter what order the rows are in each source file
* it makes the join key explicit and visible in the code
* `how='left'` keeps every referral row and attaches waiting-time figures where they exist, mirroring what `concat` did, but doing so safely.

In [21]:
imaging_df_merged = (
    nrefs
    .merge(rtt, on=['region', 'org_code', 'imaging_type'], how='left')
    .merge(ttr, on=['region', 'org_code', 'imaging_type'], how='left')
)
imaging_df_merged.head()

,region,org_code,provider_x,imaging_type,n_referrals,provider_y,mdn_days_rtt,provider,mdn_days_ttr
0,Y56,NT9,Alliance Medical,Computerized Axial Tomography,5145,Alliance Medical,1.0,Alliance Medical,5.0
1,Y56,NT9,Alliance Medical,Diagnostic Ultrasonography,2495,Alliance Medical,10.0,Alliance Medical,0.0
2,Y56,NT9,Alliance Medical,Magnetic Resonance Imaging,18740,Alliance Medical,6.0,Alliance Medical,3.0
3,Y56,NT9,Alliance Medical,Nuclear Medicine Procedure,<NA>,Alliance Medical,NaN,Alliance Medical,NaN
4,Y56,NT9,Alliance Medical,Plain Radiography,990,Alliance Medical,50.0,Alliance Medical,NaN


In [22]:
imaging_df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1325 entries, 0 to 1324
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   region        1325 non-null   object 
 1   org_code      1325 non-null   object 
 2   provider_x    1325 non-null   object 
 3   imaging_type  1325 non-null   object 
 4   n_referrals   1303 non-null   Int32  
 5   provider_y    1325 non-null   object 
 6   mdn_days_rtt  1273 non-null   float64
 7   provider      1325 non-null   object 
 8   mdn_days_ttr  1281 non-null   float64
dtypes: Int32(1), float64(2), object(6)
memory usage: 89.4+ KB


In [23]:
imaging_df_merged.shape

(1325, 9)

## What happens if the rows are not in the same order?

So far we have claimed that `pd.concat(axis=1)` is risky, but we haven't actually *seen* it fail. 

The datasets we downloaded happen to be in the same order, so our concatenation worked. 
That is exactly what makes this approach risky: it works... until it doesn't.

Real-world data is rarely so cooperative. A provider might be missing from one file, rows might 
be sorted differently after an update, or a dataset might be re-issued in a new order. 
`pd.concat()` will happily glue the rows together in whatever order they arrive, with **no 
warning at all**.

Let's simulate that scenario. We will shuffle the rows of `rtt` using `.sample(frac=1).reset_index(drop=True)` (which returns 
a copy of the dataframe with rows in a random order) and then try both approaches:

* `pd.concat()` lines up rows by **position**
* `.merge()` lines up rows by **key**


Look carefully at the output of each. In the concatenation, check whether the `region`, 
`org_code` and `imaging_type` columns from `rtt` still agree with the ones from `nrefs` in 
the same row. Then check the median waiting times — are they attached to the right provider?

In [24]:
# shuffle the rows of rtt
rtt_shuffled = rtt.sample(frac=1, random_state=42).reset_index(drop=True)

In [25]:
# shuffled row dataset
rtt_shuffled.head()

,region,org_code,imaging_type,provider,mdn_days_rtt
0,Y61,RWG,Magnetic Resonance Imaging,West Hertfordshire Hospitals NHS Trust,34.0
1,Y59,NT2,Fluoroscopy,Nuffield Health,NaN
2,Y61,RWG,Diagnostic Ultrasonography,West Hertfordshire Hospitals NHS Trust,11.0
3,Y56,RPY,Computerized Axial Tomography,The Royal Marsden NHS Foundation Trust,22.0
4,Y63,RWY,Medical Photography,Calderdale and Huddersfield NHS Foundation Trust,0.0


In [26]:
# original dataset
rtt.head()

,region,org_code,imaging_type,provider,mdn_days_rtt
0,Y56,NT9,Computerized Axial Tomography,Alliance Medical,1.0
1,Y56,NT9,Diagnostic Ultrasonography,Alliance Medical,10.0
2,Y56,NT9,Magnetic Resonance Imaging,Alliance Medical,6.0
3,Y56,NT9,Nuclear Medicine Procedure,Alliance Medical,NaN
4,Y56,NT9,Plain Radiography,Alliance Medical,50.0


In [27]:
# concat: lines up rows by position - silently wrong!
pd.concat([nrefs, rtt_shuffled], axis=1).head()

,region,org_code,provider,imaging_type,n_referrals,region,org_code,imaging_type,provider,mdn_days_rtt
0,Y56,NT9,Alliance Medical,Computerized Axial Tomography,5145,Y61,RWG,Magnetic Resonance Imaging,West Hertfordshire Hospitals NHS Trust,34.0
1,Y56,NT9,Alliance Medical,Diagnostic Ultrasonography,2495,Y59,NT2,Fluoroscopy,Nuffield Health,NaN
2,Y56,NT9,Alliance Medical,Magnetic Resonance Imaging,18740,Y61,RWG,Diagnostic Ultrasonography,West Hertfordshire Hospitals NHS Trust,11.0
3,Y56,NT9,Alliance Medical,Nuclear Medicine Procedure,<NA>,Y56,RPY,Computerized Axial Tomography,The Royal Marsden NHS Foundation Trust,22.0
4,Y56,NT9,Alliance Medical,Plain Radiography,990,Y63,RWY,Medical Photography,Calderdale and Huddersfield NHS Foundation Trust,0.0


In [28]:
# merge: lines up rows by keys - still correct
nrefs.merge(rtt_shuffled, on=['region', 'org_code', 'imaging_type']).head()

,region,org_code,provider_x,imaging_type,n_referrals,provider_y,mdn_days_rtt
0,Y56,NT9,Alliance Medical,Computerized Axial Tomography,5145,Alliance Medical,1.0
1,Y56,NT9,Alliance Medical,Diagnostic Ultrasonography,2495,Alliance Medical,10.0
2,Y56,NT9,Alliance Medical,Magnetic Resonance Imaging,18740,Alliance Medical,6.0
3,Y56,NT9,Alliance Medical,Nuclear Medicine Procedure,<NA>,Alliance Medical,NaN
4,Y56,NT9,Alliance Medical,Plain Radiography,990,Alliance Medical,50.0


## Final step : create a South West dataset

In [29]:
# these are the south west org codes.
south_west = ['RB2', 'RK9', 'RA9', 'REF', 'RH8']
sw_imaging = imaging_df_merged[imaging_df_merged['org_code'].isin(south_west)]
sw_imaging.head(3)

,region,org_code,provider_x,imaging_type,n_referrals,provider_y,mdn_days_rtt,provider,mdn_days_ttr
275,Y58,REF,Royal Cornwall Hospitals NHS Trust,Computerized Axial Tomography,46160,Royal Cornwall Hospitals NHS Trust,3.0,Royal Cornwall Hospitals NHS Trust,0.0
276,Y58,REF,Royal Cornwall Hospitals NHS Trust,Diagnostic Ultrasonography,72985,Royal Cornwall Hospitals NHS Trust,14.0,Royal Cornwall Hospitals NHS Trust,0.0
277,Y58,REF,Royal Cornwall Hospitals NHS Trust,Fluoroscopy,12320,Royal Cornwall Hospitals NHS Trust,0.0,Royal Cornwall Hospitals NHS Trust,0.0


In [30]:
sw_imaging.shape

(31, 9)

In [31]:
sw_imaging.to_csv('sw_imaging.csv', index=False)